# Работа с Excel из Python: openpyxl и XlsxWriter

**Сквозная задача:** открыть существующую книгу интернет-магазина, рассчитать помесячную выручку и создать новый оформленный Excel-отчёт.


## 1. Зачем автоматизировать Excel

Excel часто используют для отчётов, списков заказов и финансовых расчётов. При регулярной работе возникают повторяющиеся действия:

- открыть файл и проверить структуру;
- отфильтровать нужные строки;
- рассчитать показатели;
- перенести результат в новый отчёт;
- применить одинаковое оформление и построить диаграмму.

Python позволяет описать эти действия один раз и затем повторять их для новых данных. Это снижает количество ручных операций и делает расчёт воспроизводимым.

> **Что сказать:** автоматизация особенно полезна, когда отчёт строится каждую неделю или каждый месяц по одной схеме.

## 2. Как Python представляет Excel-книгу

Файл `.xlsx` представляет собой ZIP-контейнер с XML-файлами. Работать с XML напрямую не требуется: библиотеки предоставляют объектную модель.

```text
Workbook — вся книга
└── Worksheet — отдельный лист
    └── Cell — ячейка
        ├── value — значение или формула
        └── style — формат, шрифт, границы и выравнивание
```

Обычный жизненный цикл программы:

```text
открыть или создать книгу → выбрать лист → прочитать или записать данные → сохранить файл
```

Координаты Excel начинаются с единицы: `A1` — первая строка и первый столбец.

## 3. openpyxl: работа с существующей книгой

`openpyxl` умеет читать и записывать файлы `.xlsx` и `.xlsm`. Основные сценарии:

- загрузить готовую книгу через `load_workbook()`;
- получить лист по имени;
- прочитать ячейку или диапазон;
- пройти по строкам через `iter_rows()`;
- изменить значения, формулы и оформление;
- сохранить книгу методом `save()`.

### Важный нюанс формул

`openpyxl` может прочитать или записать формулу, например `=SUM(A1:A10)`, но не заменяет вычислительный движок Excel. Обычно формулу пересчитывает Excel при открытии файла.

- `data_only=False` возвращает текст формулы;
- `data_only=True` возвращает последнее сохранённое вычисленное значение, если оно существует.

> **Что сказать:** эта библиотека подходит, когда исходный Excel-файл уже существует и его нужно прочитать или изменить.

## 4. XlsxWriter: создание нового отчёта

XlsxWriter последовательно формирует новую книгу. Библиотека особенно удобна для отчётов, в которых нужны:

- числовые форматы;
- цвета, шрифты, границы и выравнивание;
- формулы Excel;
- условное форматирование;
- диаграммы;
- несколько оформленных листов.

Жизненный цикл:

```text
Workbook() → add_worksheet() → write() → add_chart() → close()
```

Главное ограничение: XlsxWriter **не читает и не изменяет существующие книги**. Он создаёт новый `.xlsx`.

## 5. Как выбрать библиотеку

| Задача | openpyxl | XlsxWriter |
|---|:---:|:---:|
| Прочитать существующую книгу | Да | Нет |
| Изменить готовый шаблон | Да | Нет |
| Создать новую книгу | Да | Да |
| Сделать оформленный отчёт | Да | Да, это основной сценарий |
| Построить диаграмму | Да | Да |

В этом примере библиотеки дополняют друг друга:

```text
online_store_dataset.xlsx
        │
        ├── openpyxl читает строки и считает выручку
        │
        └── XlsxWriter создаёт monthly_sales_report.xlsx
```

## 6. Практическая задача и исходные данные

Книга `online_store_dataset.xlsx` содержит четыре листа:

- `Описание` — краткая информация о датасете;
- `Заказы` — 220 заказов за январь–июнь 2026 года;
- `Товары` — справочник из 15 товаров;
- `Клиенты` — справочник из 36 клиентов.

Мы рассчитаем помесячную выручку только для заказов со статусом `Выполнен`.

Для одной строки:

$$
\text{выручка} = \text{количество} \cdot \text{цена} \cdot (1 - \text{скидка})
$$

Затем XlsxWriter запишет итог по месяцам, добавит формулу среднего чека и построит столбчатую диаграмму.

In [2]:
from collections import defaultdict
from pathlib import Path

import openpyxl
import xlsxwriter

candidates = [
    Path("online_store_dataset.xlsx"),
]
DATA_FILE = next((path for path in candidates if path.exists()), candidates[0])
REPORT_FILE = DATA_FILE.parent / "monthly_sales_report.xlsx"

assert DATA_FILE.exists(), f"Файл не найден: {DATA_FILE.resolve()}"

print("Исходная книга:", DATA_FILE)
print("Будущий отчёт:", REPORT_FILE)
print("openpyxl:", openpyxl.__version__)
print("XlsxWriter:", xlsxwriter.__version__)

Исходная книга: online_store_dataset.xlsx
Будущий отчёт: monthly_sales_report.xlsx
openpyxl: 3.1.5
XlsxWriter: 3.2.9


### 6.1. Открываем книгу через openpyxl

`load_workbook()` читает файл и возвращает объект `Workbook`. Пока мы не вызываем `save()`, исходная книга не изменяется.

In [3]:
workbook = openpyxl.load_workbook(DATA_FILE, data_only=False)

print("Листы:", workbook.sheetnames)
for sheet in workbook.worksheets:
    print(f"{sheet.title}: {sheet.max_row} строк, {sheet.max_column} столбцов")

Листы: ['Описание', 'Заказы', 'Товары', 'Клиенты']
Описание: 9 строк, 6 столбцов
Заказы: 221 строк, 13 столбцов
Товары: 16 строк, 6 столбцов
Клиенты: 37 строк, 6 столбцов


### 6.2. Читаем строки листа

Есть два распространённых способа обращения к данным:

```python
sheet["A1"].value              # одна ячейка
sheet.iter_rows(values_only=True) # последовательный обход строк
```

Параметр `values_only=True` возвращает обычные значения Python вместо объектов `Cell`. Для анализа строк это удобнее.

In [4]:
orders_sheet = workbook["Заказы"]
headers = [cell.value for cell in orders_sheet[1]]

print("Колонки:", ", ".join(headers))
print("\nПервые пять заказов:")

for values in orders_sheet.iter_rows(min_row=2, max_row=6, values_only=True):
    order = dict(zip(headers, values))
    price = f"{order['unit_price_rub']:,.0f}".replace(",", " " )
    print(
        f"{order['order_id']} | {order['order_date']:%Y-%m-%d} | "
        f"{order['product_id']} | {order['city']} | "
        f"{order['quantity']} шт. | {price} ₽ | "
        f"{order['status']}"
    )

Колонки: order_id, order_date, customer_id, product_id, city, channel, quantity, unit_price_rub, discount_percent, promo_code, payment_method, status, manager

Первые пять заказов:
O0025 | 2026-01-02 | P007 | Тверь | 1 шт. | 10 990 ₽ | Возврат
O0128 | 2026-01-04 | P010 | Ярославль | 1 шт. | 1 190 ₽ | Возврат
O0133 | 2026-01-04 | P001 | Ярославль | 2 шт. | 6 490 ₽ | Выполнен
O0093 | 2026-01-06 | P007 | Казань | 4 шт. | 10 990 ₽ | Возврат
O0148 | 2026-01-06 | P005 | Великий Новгород | 2 шт. | 7 490 ₽ | Выполнен


## 7. Рассчитываем помесячную выручку

Алгоритм состоит из пяти шагов:

1. Перебрать строки листа `Заказы`.
2. Сопоставить значения строки с названиями столбцов.
3. Оставить только заказы со статусом `Выполнен`.
4. Рассчитать выручку строки с учётом скидки.
5. Добавить количество заказов и выручку к соответствующему месяцу.

`defaultdict` автоматически создаёт начальный накопитель для нового месяца.

In [6]:
monthly = defaultdict(lambda: {"orders": 0, "revenue": 0.0})

for values in orders_sheet.iter_rows(min_row=2, values_only=True):
    order = dict(zip(headers, values))

    if order["status"] != "Выполнен":
        continue

    month = order["order_date"].strftime("%Y-%m")
    revenue = (
        order["quantity"]
        * order["unit_price_rub"]
        * (1 - order["discount_percent"])
    )

    monthly[month]["orders"] += 1
    monthly[month]["revenue"] += revenue

summary = [
    (month, values["orders"], round(values["revenue"], 2))
    for month, values in sorted(monthly.items())
]

print("Месяц    Заказы    Выручка")
for month, order_count, revenue in summary:
    print(f"{month} {order_count:7d} {revenue:14,.2f} ₽".replace(",", " "))

total_orders = sum(row[1] for row in summary)
total_revenue = sum(row[2] for row in summary)
print("-" * 32)
print(f"Итого:   {total_orders:7d} {total_revenue:14,.2f} ₽".replace(",", " "))

Месяц    Заказы    Выручка
2026-01      25     437 288.80 ₽
2026-02      15     347 551.90 ₽
2026-03      24     283 362.60 ₽
2026-04      27     394 376.40 ₽
2026-05      23     436 890.00 ₽
2026-06      16     217 194.40 ₽
--------------------------------
Итого:       130   2 116 664.10 ₽


## 8. Создаём новый отчёт через XlsxWriter

Теперь исходную книгу читать больше не нужно: в `summary` уже находятся готовые результаты.

Новый отчёт будет содержать:

- название;
- таблицу по месяцам;
- денежные форматы;
- формулу среднего чека;
- столбчатую диаграмму выручки.

Формат создаётся один раз методом `add_format()` и затем повторно применяется к нужным ячейкам. Формула записывается как обычная формула Excel.

In [7]:
report = xlsxwriter.Workbook(REPORT_FILE)
sheet = report.add_worksheet("Помесячный отчёт")

title_format = report.add_format({
    "bold": True,
    "font_size": 16,
    "font_color": "#1F4E78",
})
header_format = report.add_format({
    "bold": True,
    "font_color": "white",
    "bg_color": "#1F4E78",
    "border": 1,
    "align": "center",
})
money_format = report.add_format({"num_format": '#,##0 "₽"'})
integer_format = report.add_format({"num_format": "0"})
average_format = report.add_format({"num_format": '#,##0 "₽"'})

sheet.write("A1", "Продажи интернет-магазина по месяцам", title_format)
sheet.write_row("A3", ["Месяц", "Заказы", "Выручка", "Средний чек"], header_format)

for row_index, (month, order_count, revenue) in enumerate(summary, start=3):
    excel_row = row_index + 1
    sheet.write(row_index, 0, month)
    sheet.write_number(row_index, 1, order_count, integer_format)
    sheet.write_number(row_index, 2, revenue, money_format)
    sheet.write_formula(
        row_index, 3, f"=C{excel_row}/B{excel_row}",
        average_format, revenue / order_count,
    )

sheet.set_column("A:A", 14)
sheet.set_column("B:B", 12)
sheet.set_column("C:D", 18)
sheet.freeze_panes(3, 0)

chart = report.add_chart({"type": "column"})
chart.add_series({
    "name": "Выручка",
    "categories": ["Помесячный отчёт", 3, 0, 3 + len(summary) - 1, 0],
    "values": ["Помесячный отчёт", 3, 2, 3 + len(summary) - 1, 2],
    "fill": {"color": "#5B9BD5"},
})
chart.set_title({"name": "Помесячная выручка"})
chart.set_x_axis({"name": "Месяц"})
chart.set_y_axis({"name": "Выручка, руб.", "num_format": '#,##0 "₽"'})
chart.set_legend({"none": True})
sheet.insert_chart("F3", chart, {"x_scale": 1.25, "y_scale": 1.15})

report.close()
print("Создан отчёт:", REPORT_FILE)

Создан отчёт: monthly_sales_report.xlsx


### 8.1. Проверяем результат

Созданный XlsxWriter файл снова открываем через `openpyxl`. Такая проверка подтверждает, что книга читается, формула сохранилась, а диаграмма присутствует.

In [8]:
check = openpyxl.load_workbook(REPORT_FILE, data_only=False)
check_sheet = check["Помесячный отчёт"]

print("Листы отчёта:", check.sheetnames)
print("Формула среднего чека в D4:", check_sheet["D4"].value)
print("Количество диаграмм:", len(check_sheet._charts))
print("Размер файла:", REPORT_FILE.stat().st_size, "байт")

Листы отчёта: ['Помесячный отчёт']
Формула среднего чека в D4: =C4/B4
Количество диаграмм: 1
Размер файла: 8388 байт


## 9. Ограничения и типичные ошибки

### Формулы
`openpyxl` и XlsxWriter записывают формулы, но полноценный пересчёт обычно выполняет Excel или другая табличная программа.

### Перезапись файла
Сохранение по существующему пути может заменить исходную книгу. Для экспериментов безопаснее использовать новое имя.

### Даты и числа
Дату следует хранить как `date` или `datetime`, а число — как число. Строка `"10 000 ₽"` выглядит как деньги, но Excel не сможет корректно суммировать её. Денежный знак задаётся числовым форматом.

### Производительность
Для больших книг у `openpyxl` существуют режимы `read_only=True` и `write_only=True`. Они уменьшают расход памяти, но ограничивают часть операций.

### Формат `.xls`
Обе библиотеки ориентированы на современный формат `.xlsx`. Старый бинарный формат `.xls` требует других инструментов или предварительного преобразования.

## 10. Выводы

1. `openpyxl` подходит для чтения и изменения существующих Excel-книг.
2. XlsxWriter удобно использовать для новых отчётов с форматами и диаграммами.
3. В одной задаче библиотеки могут дополнять друг друга.
4. Значение ячейки, формула и её оформление — разные свойства.
5. Автоматический отчёт можно повторно построить при появлении новых исходных данных.
   